NOTEBOOK DIFFERS FROM PY FILES IN THE IMPORTS (CELL TITLE "SETUP") AND THE FOV SELECTION CODE (CELL TITLE "CROPPING FUNCTIONS") and not looping over recordings and region_extraction, and also using some uses plotting functions

In [1]:
#@title setup

virtenv = 'caiman' # 'deepcad' for denoising or 'caiman' for everything else

from ScanImageTiffReader import ScanImageTiffReader

import mat73

from natsort import natsorted

import glob
import os
import fnmatch
import logging
import sys
import re
import shutil
import datetime
from ast import literal_eval

import numpy as np
from numpy.core.records import fromarrays
import random

from tqdm import tqdm
import scipy
from scipy import signal
import scipy.io as sio

from tifffile.tifffile import imwrite, imread

import matplotlib.pyplot as plt
from matplotlib.widgets  import RectangleSelector
import matplotlib.animation as animation

from skimage.util.dtype import img_as_float
import plotly.express as px

from dash import dcc # was import dash_core_components as dcc #!pip install dash-core-components
from dash import html # was import dash_html_components as html #!pip install dash-html-components
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash

import time

import cv2

try:
    cv2.setNumThreads(0)
except:
    pass


match virtenv:

    case 'caiman':

        import bokeh.plotting as bpl
        import h5py
        import ipyparallel

        import caiman as cm
        import caiman.source_extraction.cnmf as cnmf
        from caiman.utils.visualization import nb_view_patches3d, plot_contours, nb_view_patches, nb_plot_contour


        ipyparalell?

        try:
            if __IPYTHON__:
                # this is used for debugging purposes only. allows to reload classes when changed
                get_ipython().magic('load_ext autoreload')
                get_ipython().magic('autoreload 2')
        except NameError:
            pass


        bpl.output_notebook()


        logging.basicConfig(format=
                                "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                            # filename="/tmp/caiman.log",
                            #level=logging.INFO,
                            )


    case 'deepcad':

        import torch

        if torch.cuda.is_available():
            print('\033[1;31mGPU accessiable. Use GPU for computation.\033[0m')
            gpu_id = torch.cuda.current_device()
            total_memory = torch.cuda.get_device_properties(gpu_id).total_memory/1024/1024
            alloc_memory = torch.cuda.memory_allocated(0)/1024/1024
            print('GPU ID: ', gpu_id, '|', torch.cuda.get_device_name(), \
                '| Memory: {:.0f} MB'.format(total_memory))
            #nvcc--version
            print('PyTorch version: ', torch.__version__)
        else:
            print('\033[1;31mNo GPU support. Please enable GPUs for the notebook:\033[0m')
            print(' 1. Navigate to Edit → Notebook Settings')
            print(' 2. Select GPU from the Hardware Accelerator drop-down')

        from deepcad.train_collection import training_class
        from deepcad.test_collection import testing_class
        from deepcad.movie_display import display, display_img
        from deepcad.utils import get_first_filename, download_demo

        #!pip install --upgrade onnx

        import yaml

        from csbdeep.utils import normalize #this won't work unless above is already installed??


2023-09-24 10:24:15.613964: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Object `ipyparalell` not found.


/var/folders/l9/cx2p5jb916q6gnl7zy00tcg00000gn/T/ipykernel_67773/2822123185.py:72: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('load_ext autoreload')
/var/folders/l9/cx2p5jb916q6gnl7zy00tcg00000gn/T/ipykernel_67773/2822123185.py:73: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('autoreload 2')


Loading BokehJS ...

In [2]:
#@title map2params


class map2params():
    def __init__(self):

        self.params = {}
        self.params['m2p_merge_thresh'] = [.9]
        self.params['m2p_gsig'] = [2] #[2, 3, 4]
        self.params['m2p_nb'] = [1]
        self.params['SC_sigma'] = [1]
        self.params['lambda_gnmf'] = [1]
        self.params['perc_baseline_snmf'] = [50] #10, 20, 40]
        self.params['max_iter_snmf'] = [1000]

        self.map = []
        self.max_index = 1
        for key,val in self.params.items():
            self.outer(val)
            self.max_index *= len(val)
        self.max_index-= 1
        self.order = list(self.params.keys())

    def outer(self,l2):
        if self.map == []:
            self.map = [[f2] for f2 in l2]
        else:
            prod = []
            for f1 in self.map:
                for f2 in l2:
                    prod.append(f1+[f2])
            self.map = prod
        return None

    def map_index(self,index):
        assert 0 <= index <= self.max_index
        return self.map[index]



class map2params_t5():
    def __init__(self):

        self.params = {}
        self.params['m2p_merge_thresh'] = [.9]
        self.params['m2p_gsig'] = [1, 2, 3] #[2, 3, 4]
        self.params['m2p_nb'] = [1, 2, 3]
        self.params['SC_sigma'] = [1, 4]
        self.params['lambda_gnmf'] = [1, 4]
        self.params['perc_baseline_snmf'] = [20,40] #10, 20, 40]
        self.params['max_iter_snmf'] = [1000]

        self.map = []
        self.max_index = 1
        for key,val in self.params.items():
            self.outer(val)
            self.max_index *= len(val)
        self.max_index-= 1
        self.order = list(self.params.keys())

    def outer(self,l2):
        if self.map == []:
            self.map = [[f2] for f2 in l2]
        else:
            prod = []
            for f1 in self.map:
                for f2 in l2:
                    prod.append(f1+[f2])
            self.map = prod
        return None

    def map_index(self,index):
        assert 0 <= index <= self.max_index
        return self.map[index]



In [3]:
#@title configs


def configs(index_extraction_param_set = 'default', fnames = None, min_mov = 0,
            md = None, do_planar_extraction = None, dims_spatial_ex = 0):

    #md['dims'] is dims of original fov, dims_spatial_ex is dims of extraction fov (which may be cropped, so not necessarily the same as md['dims']

    #motion correction configs

    pw_rigid = False
    nonneg_movie = True
    min_mov = min_mov
    shifts_opencv = True #true uses intercubic interp, false uses fourier, but i think something else overrides this setting elsewhere
    upsample_factor_grid = 4 #default 4
    niter_rig = 1 #default 1
    max_deviation_rigid = 3

    if md['dims'][1]==1:
        is3D_mc = False
        indices_mc = (slice(None), slice(None)) #if is3d is true for motion correction, will overwrite with nones and will lose indices_ex
        strides_mc = (24, 24)
        overlaps_mc = (12, 12)
        max_shifts_mc = (8, 8)
    else:
        is3D_mc = True
        indices_mc = (slice(None), slice(None), slice(None)) #if is3d is true for motion correction, will overwrite with nones and will lose indices_ex
        strides_mc = (24, 24, 6)
        overlaps_mc = (12, 12, 3)
        max_shifts_mc = (8, 8, 2)

    do_slices = False #my crop_fov is meant to replace this, so should always be false
    if do_slices:
        sly = slice(6, 51, 1)
        slx = slice(40, 181, 1)
        slz = slice(1, 9, 1)
        indices_ex = [slx, sly, slz]
    else:
        indices_ex = [slice(None), slice(None), slice(None)]


    only_init = False #only use the initialization run for extraction

    p = 0                   # order of the autoregressive system - 0 for nonspiking, 1 for instanteous rise but not decay, 2 for non-ionstantaneous rise and decay
    merge_thresh = 0.85
    gSig = [2, 2, 1] #forces to be odd so gsiz min is 3 (ie gsig 0.5 is same as 1)  # gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
    nb = 1 #num background components

    fr = md['volrate'] #0.6193  #9.8465 frame period so 1000 / (9.8465 *(113+51)) # approximate frame rate of data - CONFIRMED FPS
    decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
    dxy = [md['xpix']/md['xfov'], md['ypix']/md['yfov'], md['numslice']/md['zfov']] #pixels per micron

    tsub = 1                # temporal downsampling
    ssub = 1               # spatial downsampling
    p_ssub = 1 #patch downsampling in space
    p_tsub = 1 #patch downsampling in time

    thr_method = 'nrg' #or 'max'
    maxthr = 0.1 #for  thr_method = 'max' keep pixels above this threshold
    nrgthr = 0.9999 #for  thr_method = 'nrg' keep pixels whose sorted cumsum contributes this much of total energy
    extract_cc = True #true will throw away isolated pixels of some kind

    method_init = 'graph_nmf' #'greedy_roi' #'graph_nmf' #sparse_NMF apparently has problems?? 'greedy_roi' python Caiman defaults to greedy_roi, looks for globular sources

    max_iter_snmf = 1000
    perc_baseline_snmf = 80
    alpha_snmf = 100 #default 1000 #for method_init sparseNMF
    #sigma_smooth_snmf = gSig #(2, 2, 0.5) #default 0.5 0.5 0.5

    lambda_gnmf = 1 #for method_init graphNMF
    SC_kernel = 'heat' #NOT TUNABLE NOW       # kernel for graph affinity matrix
    SC_sigma = 1              # std for SC kernel
    SC_thr = 0                 # threshold for affinity matrix
    SC_normalize = True        # standardize entries prior to computing affinity matrix
    SC_use_NN = False          # sparsify affinity matrix by using only nearest neighbors
    SC_nnn = 20                # number of nearest neighbors to use if SC_use_NN = True

    low_rank_background = True #True #true makes bankground nb, false makes it update with hals
    update_background_components = True   

    fudge_factor = 0.96        # (default is 0.96; old value = 1) -- bias correction factor for discrete time constants
    ITER = 10                # (default is 2; old value=5) -- block coordinate descent iterations
    bas_nonneg = True #True

    rolling_sum = True

    #Each parameter has a low threshold (rval_lowest (default -1), SNR_lowest (default 0.5), cnn_lowest (default 0.1))
    # and high threshold (rval_thr (default 0.8), min_SNR (default 2.5), min_cnn_thr (default 0.9)).
    # A component has to exceed ALL low thresholds as well as ONE high threshold to be accepted.
    # can turn off CNN part withy use_CNN = false
    # these values will result in no roi filtering
    SNR_lowest = 0#0.5#0  #0.5 default       # minimum SNR for accepted components
    min_SNR = 0#0  #2.5 default    # accept components with that peak-SNR or higher
    rval_lowest = -1  # -1 default 0.6  # space correlation threshold
    rval_thr = 0#0  # 0.8 default  # space correlation threshold
    use_cnn = False      # True default # use the CNN classifier affects if 2 below params are used
    cnn_lowest = 0  #0.1 default  # neurons with cnn probability lower than this value are rejected
    min_cnn_thr = 0 #0.9 default # if cnn classifier predicts below this value, reject

    #stride_to_rf_ratio (along with gSig) is for automatic calculation of rf, stride, and k, 
    # keep stride_to_rf_ratio in approxoimate range 0.3 - 0.8, give or take (don't stay >0 and <=1)
    # smaller stride_to_rf_ratio means larger patch with smaller patch overlap (caiman mistakenly calls patch overlap "stride")
    # caiman says patch dia should be 3-4 times neuron dia and patch stride should be at least neuron dia
    # this calculation is just based on the largest dim of neuron, to be conservative 
    # if gsig is very diofferent across dims you may consider a different automated calculation of rf and stride and k 
    #, example: when stride_to_rf_ratio = 0.3, patch is ~7 times larger and stride is 50% larger 
    # another exmaple: when stride_to_rf_ratio = 0.8, patch is ~3 times larger and stride is 50% larger 
    # since, patch dia = ceil(neuron_dia/stride_to_rf_ratio)+1)*2 and patch stride = ceil(ceil(neuron_dia/stride_to_rf_ratio)+1)*stride_to_rf_ratio)+1
    stride_to_rf_ratio = 0.65 #0.65 
   
    if index_extraction_param_set != 'default': #create param set whose index matches value in index_extraction_param_set
        if md['dims'][1]==1:
            print("USING ALTERNATE MAP2PARAMS FOR OLD PROJECT")
            map_index_2_params = map2params_t5()
        else:
            map_index_2_params = map2params()

        print('indexing into param set')
        merge_thresh, m2p_gsig, nb, SC_sigma, lambda_gnmf, perc_baseline_snmf, max_iter_snmf = map_index_2_params.map_index(int(index_extraction_param_set))
        gSig = [m2p_gsig, m2p_gsig, 1]  #gSiz (made from gsig) will be 2 for 0.5 or 1, so don't bother with 0.5, and forcing 3rd element (z) to be 1 because my z sections are about half the neuron diameter

    if np.all(np.array(dims_spatial_ex)<50): #dont bother with patches if FOV is small enough (but this should be adjusted for dirtier drivers)
        do_patches = False
    else:
        do_patches = True
    #do_patches = False
    #determine k in automated way based on gSig, roi_decimation_fac, and stride_to_rf_ratio, while also satisfying caiman patch size recommendations
    roi_decimation_fac = 0.1  #1 is space filling, caiman demo is effectively around .33
    if do_patches: # PROCESS IN PATCHES AND THEN COMBINE, patches are useful if activity stats vary over fov (e.g. extracting same neurons from regions with varying SNR, patch runs will adapt to local stats)

        if do_planar_extraction==True:
            maxsig = np.max(gSig[0:2])
        else:
            maxsig = np.max(gSig)

        rf = int(np.ceil((maxsig*2+1) / stride_to_rf_ratio)) + 1
        stride_cnmf = int(np.ceil(rf * stride_to_rf_ratio)) + 1
        
        p_patch = p
        nb_patch = nb
        if do_planar_extraction==True:
            k = int(np.round( (rf*2*rf*2) / ((gSig[0]*2+1)*(gSig[1]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes
            #k = int(np.round(rf*rf / (gSig[0]*gSig[1])))
        else:
            if rf*2<dims_spatial_ex[2]:
                rfz = rf*2
            else:
                rfz = dims_spatial_ex[2]
            k = int(np.round( (rf*2*rf*2*rfz) / ((gSig[0]*2+1)*(gSig[1]*2+1)*(gSig[2]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes
        indices_ex = [slice(None), slice(None), slice(None)]
    else:                   # PROCESS THE WHOLE FOV AT ONCE
        rf = None # will run CNMF on the whole FOV
        stride_cnmf = None       # will run CNMF on the whole FOV
        p_patch = p
        nb_patch = nb
        k = int(np.round( np.prod(dims_spatial_ex) / ((gSig[0]*2+1)*(gSig[1]*2+1)*(gSig[2]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes

    dimstr = "3dex"
    if do_planar_extraction==True:
        dimstr = '2dex'
        indices_ex = indices_ex[:-1] #change from 3d to 2d
        dxy = dxy[:-1] #change from 3d to 2d
        gSig = gSig[:-1] #change from 3d to 2d

    gSiz = [int(np.round(2*gstmp + 1)) for gstmp in gSig] #put here at end to register any gSig change
    se = np.ones((3,)*len(gSig), dtype=np.uint8)  #put here at end to register any gSig change #se = np.ones((3,3,1), dtype=np.uint8)
    #medw = (3,)*len(gSig)

    sigma_smooth_snmf = [0.5] #append this filter sigma for time to beginning (when it is applied time is in 1st dim?)
    sigma_smooth_snmf.extend(gSig)
    if do_planar_extraction==False:
        sigma_smooth_snmf[-1] = 0.5 #sigma_smooth_snmf can actually use values<1, if 3d extraction, make small for coarse z samples

    fnadd = str(gSig[0]) + '_' + str(nb) + '_' + str(merge_thresh) \
        + '_' + str(rf) + '_' + str(SC_sigma) + '_' + str(lambda_gnmf) + '_' + str(perc_baseline_snmf) \
        + '_' + str(max_iter_snmf) + '_' + str(k) + '_' + method_init.split('_')[0] + '_' + dimstr

    if do_planar_extraction is None: #it's none during motion correction, when we don't care about these params, rather than true/false
        print("motion correction params configured")
    else:
        print("index_extraction_param_set is " + str(index_extraction_param_set) + " with filename string " + fnadd)

    opts_dict = {'strides': strides_mc,    # start a new patch for pw-rigid motion correction every x pixels
                'overlaps': overlaps_mc,   # overlap between pathes (size of patch strides+overlaps)
                'max_shifts': max_shifts_mc,   # maximum allowed rigid shifts (in pixels)
                'max_deviation_rigid': max_deviation_rigid,  # maximum shifts deviation allowed for patch with respect to rigid shifts
                'pw_rigid': pw_rigid,         # flag for performing non-rigid motion correction
                'is3D': is3D_mc,
                'nonneg_movie':nonneg_movie,
                'min_mov': min_mov,
                'shifts_opencv':shifts_opencv,
                'niter_rig':niter_rig,
                'upsample_factor_grid':upsample_factor_grid,
                'fr': fr,
                'p': p,
                'nb': nb,
                'merge_thr': merge_thresh,
                'rf': rf,
                'indices': indices_mc,  #for some reason indices_mc is causing error, maybe needs list for mc and tuple for extraction?
                'K': k,
                'gSig': gSig,
                'gSiz': gSiz,
                'stride': stride_cnmf,
                'method_init': method_init,
                'sigma_smooth_snmf': sigma_smooth_snmf,
                'perc_baseline_snmf': perc_baseline_snmf,
                'max_iter_snmf': max_iter_snmf,
                'sigma_smooth_snmf': sigma_smooth_snmf,
                'SC_kernel': SC_kernel, #NOT TUNABLE NOW       # kernel for graph affinity matrix
                'SC_sigma': SC_sigma,            # std for SC kernel
                'SC_thr': SC_thr,                 # threshold for affinity matrix
                'SC_normalize': SC_normalize,        # standardize entries prior to computing affinity matrix
                'SC_use_NN': SC_use_NN,          # sparsify affinity matrix by using only nearest neighbors
                'SC_nnn': SC_nnn,                # number of nearest neighbors to use if SC_use_NN = True
                'lambda_gnmf': lambda_gnmf, #for method_init graphNMF
                'alpha_snmf': alpha_snmf, #for method_init sparseNMF
                #'dims': dims,
                'dxy': dxy,
                'decay_time': decay_time,
                'bas_nonneg': bas_nonneg,
                'p_patch': p_patch,
                'nb_patch': nb_patch,
                'se': se,
                'rolling_sum': rolling_sum,
                'only_init': only_init,
                'ssub': ssub,
                'tsub': tsub,
                'p_ssub': p_ssub,
                'p_tsub': p_tsub,
                'SNR_lowest': SNR_lowest,
                'min_SNR': min_SNR,
                'rval_thr': rval_thr,
                'rval_lowest': rval_lowest,
                'use_cnn': use_cnn,
                'min_cnn_thr': min_cnn_thr,
                'cnn_lowest': cnn_lowest,
                'ITER': ITER,
                'fudge_factor': fudge_factor,
                'cnn_lowest': cnn_lowest,
                'update_background_components': update_background_components,
                'low_rank_background' : low_rank_background,
                'thr_method': thr_method,
                'maxthr': maxthr,
                'nrgthr': nrgthr,
                'fnames': fnames,
                #'medw': medw,
                'extract_cc': extract_cc}


    # # for reference here are the initialization defs for 3 methods, sparse_nmf apparently "has problems" according to gitter

    # # greedyROI(Y, nr=30, gSig=[5, 5], gSiz=[11, 11], nIter=5, kernel=None, nb=1,
    #           rolling_sum=False, rolling_length=100, seed_method='auto')

    # for graphnmf all these are tunable in cnmf params except remove_baseline, truncate, tol, and SC_kernel whose defaults are below
    # note SC_kernel appears to be tunable because it's in params but it is not, heat is default
    # # graphNMF(Y_ds, nr, max_iter_snmf=500, lambda_gnmf=1,
    #          sigma_smooth=(.5, .5, .5), remove_baseline=True,
    #          perc_baseline=20, nb=1, truncate=2, tol=1e-3, SC_kernel='heat',
    #          SC_normalize=True, SC_thr=0, SC_sigma=1, SC_use_NN=False,
    #          SC_nnn=20

    # for sparsenmf all these are tunable in cnmf params except remove_baseline and truncate, whose defaults are below
    # # sparseNMF(Y_ds, nr, max_iter_snmf=500, alpha=10e2, sigma_smooth=(.5, .5, .5),
    #           remove_baseline=True, perc_baseline=20, nb=1, truncate=2)


    return opts_dict, indices_ex, fnadd




In [4]:
#@title read and save metadata


def read_save_metadata(pth_datafile, pth_md, pth_md_npy, mat_file_shape = None):

    mdt = {}

    if mat_file_shape is None:

        # use ScanImageTiffReader to read metadata (strange parsing because scanimage tif headers are not saved as json)
        meta = ScanImageTiffReader(pth_datafile).metadata()
        mdt['numvol'] = int(re.findall( 'actualNumVolumes = (.*)', meta)[0])
        mdt['numslice_withflyback'] = int(re.findall( 'numFramesPerVolumeWithFlyback = (.*)', meta)[0])
        mdt['numslice'] = int(re.findall( 'actualNumSlices = (.*)', meta)[0])
        mdt['xpix'] = int(re.findall( 'pixelsPerLine = (.*)', meta)[0])
        mdt['ypix'] = int(re.findall( 'linesPerFrame = (.*)', meta)[0])
        mdt['flyback'] = mdt['numslice_withflyback'] - mdt['numslice']
        mdt['dims'] = [mdt['numvol'], mdt['numslice_withflyback'] - mdt['flyback'], mdt['ypix'], mdt['xpix']]
        fovtmp = literal_eval(re.findall( 'imagingFovUm = (.*)', meta)[0].replace(" ",",").replace(";",","))
        mdt['xfov'] = abs(fovtmp[0]) + abs(fovtmp[2])
        mdt['yfov'] = abs(fovtmp[1]) + abs(fovtmp[3])
        mdt['zwid'] = int(re.findall( 'actualStackZStepSize = (.*)', meta)[0])
        mdt['zstartpos'] = literal_eval(re.findall( 'zsRelative = (.*)', meta)[0].replace(";",","))
        mdt['zfov'] = mdt['zstartpos'][-1] + mdt['zwid'] - mdt['zstartpos'][0]
        mdt['framerate'] = float(re.findall( 'scanFrameRate = (.*)', meta)[0])
        mdt['volrate'] = float(re.findall( 'scanVolumeRate = (.*)', meta)[0])

    else:

        mdt['dims'] = [mat_file_shape[0], 1, mat_file_shape[1], mat_file_shape[2]] #z size (2nd dim) is 1 because old project is not volumetric
        mdt['framerate'] = 20
        mdt['volrate'] = 20
        mdt['xpix'] = 256
        mdt['xfov'] = 74
        mdt['ypix'] = 128
        mdt['yfov'] = 37
        mdt['numslice'] = 1
        mdt['numslice_withflyback'] = 1
        mdt['numvol'] = mat_file_shape[0]
        mdt['zfov'] = 1  #set to 1 to avoid division by zero later, even though it's not really 1
        mdt['flyback'] = 0
        mdt['zwid'] = 0
        mdt['zstartpos'] = 0


    md = fromarrays( [ mdt['numvol'], mdt['numslice_withflyback'], mdt['numslice'], mdt['xpix'], \
        mdt['ypix'], mdt['flyback'], mdt['xfov'], mdt['yfov'], \
        mdt['zwid'], mdt['zfov'], mdt['framerate'], mdt['volrate'] ], \
        names = ['numvol', 'numslice_withflyback', 'numslice', 'xpix', 'ypix', 'flyback', \
            'xfov', 'yfov', 'zwid', 'zfov', 'framerate', 'volrate' ] )

    sio.savemat(pth_md, {'md': md}) #save for matlab part of pipeline

    with open(pth_md_npy, 'wb') as fnmd: #and save as npy file for rest of python pipeline
        np.save(fnmd, mdt)

    return mdt


In [5]:
#@title cropping functions

def select_fov_z(images):

  %matplotlib inline

  numim = images.shape[-1]

  Nc = 3
  Nr = int(np.ceil(numim/Nc))
  fig, ax = plt.subplots(Nr, Nc)
  for i in range(numim):
      indies = np.unravel_index(i, (Nr,Nc))
      ax[indies[0],indies[1]].imshow(images[:,:,i].T)
      ax[indies[0],indies[1]].axis('off')
  interval = 5 # units of seconds
  plt.pause(interval)
  plt.show()
  print("what z slices do you want to keep? consider keeping first as padding if cells abut z edges")
  zlimits = literal_eval(input ("choose z limits, format (firstframe,lastframe) one-indexed: "))
  return zlimits



def select_fov_xy(img):

  img = img.T
  fig = px.imshow(img)
  fig_hist = px.histogram(img.ravel())

  fig.update_layout(
      autosize=False,
      # width=500,
      height=400,
      dragmode='drawrect',
      newshape=dict(line_color='cyan', line_width=0.5))

  # this doesn't work: plots a second figure that is editable but lacks propertiers of the other
  # fig.show(config={
  #           'editable': True,
  #           'edits': {'shapePosition': True}})

  app = JupyterDash(__name__)
  app.layout = html.Div(
      [
          html.H3("Drag a rectangle to show the histogram of the ROI"),
          html.Div([dcc.Graph(id="middle-sai", figure=fig)], style={"width": "60%", "display": "inline-block", "padding": "0 0"}),
          html.Pre(id="bbox-data", style={"width": "10%", "display": "inline-block", "padding": "0 0"}),
          #html.Div(id='all-bboxes',style={"color": "white"}),
          html.Div([dcc.Graph(id="histogram", figure=fig_hist)], style={"width": "30%", "display": "inline-block", "padding": "0 0"}),
      ]
  )

  @app.callback(
      Output("histogram", "figure"),
      Output("bbox-data", "children"),
      Input("middle-sai", "relayoutData"),
      prevent_initial_call=True,
  )
  def on_new_annotation(relayout_data):

      if "shapes" in relayout_data:

          last_shape = relayout_data["shapes"][-1]
          x1, y0 = int(last_shape["x0"]), int(last_shape["y0"]) #slicing requires ints
          x0, y1 = int(last_shape["x1"]), int(last_shape["y1"]) #slicing requires ints
          roi_img = img[y0:y1, x0:x1]
          coords = (y0, y1, x0, x1)
          croplimxy = np.asarray(coords).astype(int)
          fn_crop_lim_xy = pth_tif_reg[0][:-4] + 'croplimxytmp_.npy'
          with open(fn_crop_lim_xy, 'wb') as fncrop:
              np.save(fncrop, croplimxy)
          return px.histogram(roi_img.ravel()),  html.Div(str(coords), style={"color": "white"})

      else:
          return no_update

  app.run()




In [6]:
#@title bgremover


class bgremover:

    def __init__(self, img, pth_save_prefix, half_wid=12, dimorder='tyx'):
        self.half_wid = half_wid
        self.pth_save_prefix = pth_save_prefix
        if dimorder=='tyx':
            self.img = img
        elif dimorder=='txy':
            self.img = np.transpose(img, (0, 2, 1))

    def draw_patches(self):
        half_wid = self.half_wid
        wid = 2*half_wid
        kernel = np.ones(wid)/wid
        self.meanframe = np.mean(self.img, axis=0)
        bg_ind = []
        for line in self.meanframe:
            tmp = np.convolve(line, kernel, 'valid')
            bg_center = np.argmin(tmp) + half_wid
            bg_ind.append([bg_center-half_wid, bg_center+half_wid])
        self.bg_ind = bg_ind

    def remove_bg(self, offset=0):
        bg_ind = self.bg_ind
        out = self.img.copy()
        for ind in range(out.shape[1]):
            patch = self.img[:, ind, :]
            bg_patch = self.img[:, ind, bg_ind[ind][0]:bg_ind[ind][1]]
            bg = bg_patch.mean(axis=-1)
            patch = patch-bg[None].T
            out[:, ind, :] = patch
        out = out + offset # compensate so that most of the pixels are above 0 (deprecated)
        self.out = out

    def make_plots(self):

        half_wid = self.half_wid

        patch_im = np.mean(self.img, axis=0)
        mv = np.max(patch_im)
        for i in range(patch_im.shape[0]):
            patch_im[i, self.bg_ind[i][0]:self.bg_ind[i][1]] = mv

        y_len, x_len = self.out.shape[1:3]
        test_y = random.randint(0, y_len-1)
        test_x = random.randint(0, x_len-2*half_wid-1) + half_wid

        test_patch = self.img[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        fs = self.img.shape[-1]
        f, Pxx_den = signal.periodogram(test, fs)
        fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8)) = plt.subplots(4, 2)
        ax1.semilogy(f[1:], Pxx_den[1:]) #skip 0 frequency

        test_patch = self.out[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        f, Pxx_den = signal.periodogram(test, fs)
        ax2.semilogy(f[1:], Pxx_den[1:]) #skip 0 frequency

        ax3.imshow(self.meanframe)
        ax3.axis('off')

        self.meanframe_nobg = np.mean(self.out, axis=0)

        ax4.imshow(self.meanframe_nobg)
        ax4.axis('off')

        ax5.hist(self.img.ravel(), bins=50)
        ax5.set_yscale('log')
        ax6.hist(self.out.ravel(), bins=50)
        ax6.set_yscale('log')

        ax7.imshow(patch_im)

        plt.savefig(self.pth_save_prefix + '_bgplots.png')
        plt.close()




In [7]:
#@title denoised plotting function

def plot_denoised_movie(filename = None, pthsave = None, filename_add = 'add', cropflag = 0, display_length = 20, norm_min_percent = 0, norm_max_percent = 100):

  %matplotlib inline

  print(filename)
  sliceind = int(filename.split('/')[-1].split('_')[3])

  fn_savegif = pthsave + '/' + filename.split('/')[-1] + '_' + filename_add + '_.mp4'
#  fn_savegif = filename[:-4] + '_' + filename_add + '_.mp4'
  Ynew = imread(filename).astype(denoise_dtype)
  print(Ynew.shape)

  #img = normalize(Ynew, norm_min_percent, norm_max_percent)


  fig, ax = plt.subplots()
  ims = []
  if cropflag:
    fuk = range(49,350)
  else:
    fuk = range(Ynew.shape[0])

  for i in fuk:
    im = ax.imshow(Ynew[i,:,:], animated=True)
    if i == 0:
        ax.imshow(Ynew[i,:,:]) # show an initial one first
    ims.append([im])

  ani = animation.ArtistAnimation(fig, ims, interval=200, repeat=True, blit=True, repeat_delay=1000) #originally repeat false
  ani.save(fn_savegif)
  # writer = animation.FFMpegWriter(fps=15, metadata=dict(artist='Me'), bitrate=1800)
  # ani.save(fn_savegif, writer=writer)

  #plt.show()

In [8]:
#@title concatenate denoised movie in z and save

def stitch_denoised_slices(pth_denoising, fn_prefix, pth_out, dims_pre_denoise, denoise_volume, epoch_choose):

  if denoise_volume == 1:
    pth_trainset_all = natsorted(glob.glob(os.path.join(pth_denoising, fn_prefix + '_all/')))
  else:
    pth_trainset_all = natsorted(glob.glob(os.path.join(pth_denoising, fn_prefix + '_*/')))
    pth_trainset_all = list(set(pth_trainset_all) - set(natsorted(glob.glob(os.path.join(pth_denoising, fn_prefix + '_al*/'))))) #exclude the "all" folders when denoise_volume==1

  countz = 0
  for pth_trainset in pth_trainset_all:
    countz = countz + 1
    fldr_outtiff_all = natsorted(glob.glob(os.path.join(pth_trainset, 'DataFolderIs_*', 'E_*'))) #for all epochs that were used for denoising, organize tif files into single folder in 'denoised' folder
    for fldr_outtiff in fldr_outtiff_all:
      if fnmatch.fnmatch(fldr_outtiff.split('/')[-1], 'E_' + "{:02d}".format(epoch_choose) + '_Iter_*'):
        pth_denoised_singles = natsorted(glob.glob(os.path.join(fldr_outtiff, '*output.tif')))

        Y = np.zeros((dims_pre_denoise[0], dims_pre_denoise[2], dims_pre_denoise[3], dims_pre_denoise[1])) #t y x z
        for fni,f in enumerate(pth_denoised_singles): #loop over each denoised z slice and reassemble into array matching shape of original 4d volume
            print(f)
            sliceind = int(f.split('/')[-1].split('_')[3])
            Ynew = imread(f)
            if Ynew.dtype!='uint16':
                print("warning, converting type from " + str(Ynew.dtype))
                Ynew = Ynew.astype('uint16')
                if np.min(Y)<0 or np.max(Y) > 65535:
                  raise Exception("denoising have operated on uint16 for this pipeline, or adjust it")
            print(Ynew.dtype)
            print(sliceind)
            Y[:,:,:,sliceind] = Ynew

        if fni != dims_pre_denoise[1]-1:
            raise Exception("not all slices present")

        mnmv = np.min(Y)
        Y = Y - mnmv #make nonnegative before writing to uint16
        print("MIN AFTER DENOISING " + str(mnmv))
        Y = Y.astype('uint16')
        Y = np.transpose(Y, (0, 3, 1, 2)) #tzyx
        print(Y.shape)
        Y = Y.reshape(dims_pre_denoise[0] * dims_pre_denoise[1], dims_pre_denoise[2], dims_pre_denoise[3]) #(tz)yx
        print(Y.shape)
        imwrite(pth_out, Y.squeeze()) #write the registered movie as tif for use in matlab, and caiman extraction below




In [9]:
#@title choose files

print(sys.executable)
env_path = sys.path

index_extraction_param_set = 'default' #specifies the extraction param set (set is created in configs.py, which uses map2params.py to help create the param sets)
recdates = ['20230627'] #list of strings, as it appears in the directory and raw file filename (with hyphen not underscore for now), '*' for any
fly = '2' #string, fly index_extraction_param_set, '*' for any
trial = '2' #string, trial index_extraction_param_set, '*' for any
region_extraction = ['pb', 'gar', 'gal', 'no'] #['pre', 'post'] #list of strings specifying names for xy rectangular or xyz cuboid fov subregions that are passed separately to source extraction; interactive plots prompt user to define z range and draw xy rectangle; use ['fullfov'] to extract from entire FOV
do_motion_correction = 0 #caiman normCorre
do_background_subtraction = 0 #won't happen unless do_motion_correction = True
do_denoise = 0 #deepcad (from the more recent deepcadrt, although this is not real time), input must be motion_corrected
denoise_slice_index = 'all' #deepcad (from the more recent deepcadrt, although this is not real time), input must be motion_corrected
do_extraction = 1 #caiman source extraction
do_planar_extraction = 0 #caiman source extraction for each plane independently (WARNING, 3D EXTRACTION REQUIRES AT LEAST 3 ELEMENTS IN EACH DIMENSION X Y and Z, OR you must REWRITE binary_closing IN CAIMAN'S THRESHOLD_COMPONENTS)
use_background_subtracted = 0 #won't happen unless do_motion_correction = True
use_denoised = 1 #use the deepcad denoised data, or just the caiman registered data
do_cropping_session = 0 #skip everything but FOV selection for all entries in region_extraction, must have already run motion correction if use_denoised=False, or motion correction and denoising if use_denoised=True, convenient to do for many recordings at once so extraction can be run on a batch of recordings in pth_allrecs without interruption
recording_index = 'all' #if 'all', loop over all recordings matching pattern in pth_allrec, if not 'all' (can be str or int) operate on recording whose index (in sorted list of all recordings in pth_allrec) matches value in recording_index

bg_patch_halfwidth = 3 #half width of patch over which mean is computed for background subtraction (patch is a line in x)
do_plots = 0 #plots were for old version of this pipeline, and I haven't verified that plots run without error, so I leave this 0
do_cluster = 0 #leave as 0 because cluster isn't working (except on colab), and typical recordings (size 128 x 256 x 20 x 3000) don't take that long
cluster_backend = 'ipyparallel' #irrelevant if do_cluster=0

if (re.search("/Users/wienecke/", env_path[0])):
  pth_allrec = '/Users/wienecke/Documents/ambrose/stacks/'
  if do_denoise: #need gpu, don't have one locally
     raise Exception("no gpu, make do_denoise false")
elif (re.search("/home/caw846/", env_path[0])):
  pth_allrec = '/n/scratch3/users/c/caw846/stacks/'
  cluster_backend = 'SLURM' #this failed on O2, and so did cluster_backend = 'ipyparallel'
elif (re.search("/home/users/wienecke/", env_path[0])):
  pth_allrec = '/scratch/users/wienecke/stacks/'
elif (re.search('/content', env_path[0])):
  pth_allrec = '/content/drive/MyDrive/stacks/'
  do_cluster = 1 #cluster worked on colab
  index_extraction_param_set = 'default' #not set up for arguments in colab

pth_super = '/'.join(pth_allrec.split('/')[:-2])
pth_denoising = os.path.join(pth_super, 'denoising')
if not os.path.exists(pth_denoising):
    os.mkdir(pth_denoising)
pth_denoised = os.path.join(pth_super, 'denoised')
if not os.path.exists(pth_denoised):
    os.mkdir(pth_denoised)

countz = 0
for recording_date in recdates:

  pth_fldrs_pattern = pth_allrec + recording_date + '-' + fly + '_*/'
  fn_pattern = recording_date + '-' + fly + '*_trial_00' + trial + '_*.tif'
  pth_fldrs = natsorted(glob.glob(pth_fldrs_pattern))
  old_mat_files = 0
  if not pth_fldrs:  #if no matches try another filename pattern (files from previous project)
    old_mat_files = 1
    pth_fldrs_pattern = pth_allrec + recording_date + '_' + fly + '_*/'
    fn_pattern = recording_date + '_' + fly + '_' + trial + '_stackraw_.mat'
    pth_fldrs = natsorted(glob.glob(pth_fldrs_pattern))

  for pth_fldr in pth_fldrs:

    print(pth_fldr)

    pth_allfiles = natsorted(os.listdir(pth_fldr))

    for f in pth_allfiles:

      if fnmatch.fnmatch(f, fn_pattern):

        countz = countz + 1
        if recording_index == 'all' or (recording_index !='all' and countz==recording_index): #if 'all', do all files matching pattern, otherwise only file matching index

          # tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
          # sys.stdout = open(pth_fldr + '/' + tmpdate + '.txt', 'w')
          # sys.stderr = sys.stdout

          pth_datafile = pth_fldr + f
          print(pth_datafile)

          if old_mat_files:
            fn_prefix = '_'.join(f.split('_')[:3])
          else:
            fn_prefix = f.split('_')[0].split('-')[0] + '_' + f.split('_')[0].split('-')[1]  + '_' + f.split('_')[-2][-1] #change hyphen to underscore

          pth_prefix = pth_fldr + fn_prefix
          if do_background_subtraction or (use_background_subtracted and not do_motion_correction):
            pth_tif_dn = pth_prefix + '_cmrg_bksb_dcdn_.tif'
            pth_tif_reg_tmp = pth_prefix + '_cmrg_bksb_tmp_.tif'
            pth_tif_reg = pth_prefix + '_cmrg_bksb_.tif'
          else:
            pth_tif_dn = pth_prefix + '_cmrg_dcdn_.tif'
            pth_tif_reg_tmp = pth_prefix + '_cmrg_tmp_.tif'
            pth_tif_reg = pth_prefix + '_cmrg_.tif'

          pth_md = pth_prefix + '_metadatanew_.mat'
          pth_md_npy = pth_md[:-4] + '.npy'

          if old_mat_files: #for my old project

            pth_datafile_new = pth_datafile[:-4] + '.tif'

            if os.path.isfile(pth_md_npy) and os.path.isfile(pth_datafile_new): #
              md = np.load(pth_md_npy, allow_pickle='TRUE').item() #if it exists, the md file will too
              pth_datafile = pth_datafile_new
            else:
              mat = mat73.loadmat(pth_datafile)
              Y = mat['stackRaw_pmc'] # Y = mat['stackRaw_mc']
              mnmv = np.min(Y)
              Y = Y - mnmv #make movie nonnegative (not sure this is necessary)
              print("MIN OF STACKRAW_PMC MAT FILE " + str(mnmv))
              Y = np.transpose(Y, (2, 0, 1)) #put in order t y x (not t x y) #stackraw_mc may be flipped relative to stackraw pmc
              pth_datafile = pth_datafile_new
              imwrite(pth_datafile, Y.astype('uint16')) #write as t x y z (singleton z at end)
              md = read_save_metadata(pth_datafile, pth_md, pth_md_npy, mat_file_shape = Y.shape)

          else:

            fn_prefix = f.split('_')[0].split('-')[0] + '_' + f.split('_')[0].split('-')[1]  + '_' + f.split('_')[-2][-1] #change hyphen to underscore
            if os.path.isfile(pth_md_npy):
               md = np.load(pth_md_npy, allow_pickle='TRUE').item()
            else:
               md = read_save_metadata(pth_datafile, pth_md, pth_md_npy, mat_file_shape = None)



/Users/wienecke/mambaforge/envs/caiman/bin/python
/Users/wienecke/Documents/ambrose/stacks/20230627-2_D05_syt7f_018_syt7f/
/Users/wienecke/Documents/ambrose/stacks/20230627-2_D05_syt7f_018_syt7f/20230627-2_D05_syt7f_018_syt7f_181750_trial_002_00001.tif


In [10]:
#@title load unregistered movie

n_processes = 1
dview = None

if do_motion_correction and not do_cropping_session:

    Y = imread(pth_datafile).astype('float32') ##having trouble on O2 with caiman function cm.load so just using imread from tifffile.tifffile

    Y = Y.reshape(md['dims'][0], md['dims'][1]+md['flyback'], md['dims'][2], md['dims'][3])

    if md['flyback']!=0:
        Y = Y[:,:-md['flyback'],:,:] #crop md['flyback'] frames

    Y = np.transpose(Y, (0, 3, 2, 1)) #put in order t x y z

    mnmv = np.min(Y)
    Y = Y - mnmv #make movie nonnegative (not sure this is necessary)
    print("MIN BEFORE MOTION CORRECTION " + str(mnmv))

    if do_background_subtraction:

        if Y.shape[3]==1: #if it's not volumetric
            zindall = [0]
        else:
            zindall = np.arange(Y.shape[3])

        for zind in zindall: #for every z slice

            dimorder = 'txy'
            pth_bgplot_save = pth_prefix + '_' + str(zind)
            br = bgremover(Y[:,:,:,zind], pth_bgplot_save, half_wid=bg_patch_halfwidth, dimorder=dimorder)
            br.draw_patches()
            br.remove_bg()
            br.make_plots()
            Y[:,:,:,zind] = np.transpose(br.out, (0, 2, 1))

        mnmv = np.min(Y)
        Y = Y - mnmv #make movie nonnegative (not sure this is necessary)
        print("MIN BEFORE MOTION CORRECTION AFTER BG SUB" + str(mnmv))


    imwrite(pth_tif_reg_tmp, Y.squeeze()) #write as t x y z (z might be singleton for non-volumetric data, so squeeze)


In [11]:
#@title start cluster

if do_cluster:
  if 'dview' in locals(): cm.stop_server(dview=dview)
  cc, dview, n_processes = cm.cluster.setup_cluster(backend=cluster_backend, n_processes=None, single_thread=False)


In [12]:
#@title register movie (motion correct)

if do_motion_correction and not do_cropping_session:


  min_mov = np.min(Y)
  opts_dict, indices_ex, fnadd = configs(index_extraction_param_set = 'default', fnames = pth_tif_reg_tmp, min_mov = min_mov, md = md) #configs for motion correction (will also define for extraction, but extraction params are in redefined later call to configs)
  opts = cnmf.params.CNMFParams(params_dict=opts_dict)

  #sys.setprofile(tracefunc)
  mc = cm.motion_correction.MotionCorrect([pth_tif_reg_tmp], dview=dview, **opts.get_group('motion'))
  mc.motion_correct(save_movie=True)
  os.remove(pth_tif_reg_tmp)

  border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0
  basename_memap = pth_tif_reg.split('/')[-1][:-4]
  pth_mmap_reg = cm.save_memmap(mc.mmap_file, base_name=basename_memap, order='C', border_to_0=border_to_0, dview=dview) # save in order C (motion_correct above has to save in order F)
  Y, dims_spatial_rg, dim_time_rg = cm.load_memmap(pth_mmap_reg)
  Y = np.reshape(Y.T, [dim_time_rg] + list(dims_spatial_rg), order='F')

  mnmv = np.min(Y)
  Y = Y - mnmv #make nonnegative before writing to uint16
  print("MIN AFTER MOTION CORRECTION " + str(mnmv))

  os.remove(mc.mmap_file[0]) #remove the mmap file in F order
  os.remove(pth_mmap_reg) #remove the mmap file in C order

  if md['dims'][1]==1:
      imwrite(pth_tif_reg, np.transpose(Y.astype('uint16'), (0, 2, 1)).reshape(dim_time_rg, dims_spatial_rg[1], dims_spatial_rg[0])) #write the registered movie as tif (uint16) for use in matlab, and caiman extraction below
  else:
      imwrite(pth_tif_reg, np.transpose(Y.astype('uint16'), (0, 3, 2, 1)).reshape(dim_time_rg * dims_spatial_rg[2], dims_spatial_rg[1], dims_spatial_rg[0])) #write the registered movie as tif (uint16) for use in matlab, and caiman extraction below





In [13]:
#@title denoise

# TRAIN MEANS LEARNING A MODEL THAT DESCRIBES AN INPUT MOVIE'S SIGNAL AND NOISE
# TEST MEANS PASSING A MOVIE THROUGH THAT MODEL TO DENOISE IT, THE OUTPUT FROM THE TESTING PHASE IS THE DENOISED MOVIE
# INPUT TO TRAIN AND TEST DO NOT HAVE TO BE THE SAME MOVIE (BUT I ALWAYS MAKE THEM THE SAME MOVIE)

# it's possible to resume training if interrrupted by loading most recent .pth file, but currently this is inconvenient
# because the deepcad code appends a timestamp to the .pth file's enclosing folder, so a new folder will be created when you resume
# it should still work, but you will have epochs spread across multiple folders
# however, the code below does not have this resuming training functionality yet,
# so to ensure models don't get mixed, there is a line that removes any existing training folder before training  (shutil.rmtree(pth_trainset_all[countz])

if do_denoise:

    denoise_slice_index = 'all' #'all' or list of integers
    denoise_volume = 0
    denoise_dtype = "uint16" #dtype for denoising, and writing results, but regardless, stitch_denoised_slices will write to uint16


    pth_in = pth_tif_reg
    pth_denoising = pth_denoising
    pth_denoised = pth_denoised
    fn_prefix = fn_prefix
    dims = md['dims'].copy()

    Y = imread(pth_in).astype(denoise_dtype)
    Y = Y.reshape(dims)
    Y = np.transpose(Y, (0, 2, 3, 1)) #put in order t y x z (not t x y z)

    fn_existing_denoised_slices = natsorted(glob.glob(pth_denoised + '/' + fn_prefix + '*output.tif'))
    if 0:#skip this for now until we know more, was previously this: if fn_existing_denoised_slices:
        largest_denoised_slice_index = int(fn_existing_denoised_slices[-1].split('/')[-1].split('_')[3])
        print(denoise_slice_index)
        print("updating denoise slice index bc largest existing is " + str(largest_denoised_slice_index))
        if denoise_slice_index=='all':
            zind_all_dn = np.arange(largest_denoised_slice_index + 1, Y.shape[-1])
        else:
            zind_all_dn = [x + largest_denoised_slice_index for x in denoise_slice_index]
        print(zind_all_dn)
    else:
        if denoise_slice_index == 'all':
            zind_all_dn = np.arange(dims[1])
        else:
            zind_all_dn = denoise_slice_index


    print(pth_in)
    print(zind_all_dn)

    if denoise_volume: #if training on all slices, put them all in one folder
        pth_trainset_all = ['']
        pth_testset_all = ['']
    else: #if training on subset of slices, put each subset in separate folder
        pth_trainset_all = ['']*len(zind_all_dn)
        pth_testset_all = ['']*len(zind_all_dn)


    countz = -1
    for zii in zind_all_dn: #deepcad wants 3d data, so organize slices into separate tif files, and put in one folder (if denoise_volume=1, ie train on all slices) or separate folders (if do_volume=0, ie train on z subset)

        Ynew = Y[:,:,:,zii]
        Lt, Ly, Lx = Ynew.shape #don't need to index these they should be the same for all stacks
        if countz>-1 and prev_shape != Ynew.shape:
          raise Exception("dims changed")
        prev_shape = Ynew.shape

        if denoise_volume:
          dnfolder_insert = 'all'
          countz = 0 #constant 0 because every slice goes to the same directory
        else:
          dnfolder_insert = str(zii)
          countz = countz + 1

        dnfolder = fn_prefix + '_' + dnfolder_insert
        tifname = fn_prefix + '_' + str(zii) + '_.tif'

        pth_trainset_all[countz] = pth_denoising + '/' + dnfolder #dir containing all tif files for training
        pth_testset_all[countz] = pth_trainset_all[countz] + '/' + dnfolder + '_*' #dir containing all models (.pth files) for test
        pth_tif_pdn = pth_trainset_all[countz] + '/' + tifname

        if os.path.exists(pth_trainset_all[countz]):
          if zii==0 or denoise_volume==0: #if you're on the first zii (regardless of denoise_volume value), or for all zii if denoise_volume==0
            shutil.rmtree(pth_trainset_all[countz]) #REMOVE any existing training folder before training, to ensure models don't get mixed (until "resume training" functionality is written)
        if not os.path.exists(pth_trainset_all[countz]): #don't make this "else" connected to "if" above because you have to evaluate it
          os.mkdir(pth_trainset_all[countz])

        print(pth_tif_pdn)
        imwrite(pth_tif_pdn, Ynew.astype(denoise_dtype), photometric='minisblack' ) #put the tif in the folder deepcad looks to for training data



    # ########################## TRAIN ##########################

    # # note train_datasets_size is number of 3d xyt patches to train the model on (so can exceed number of frames), and also is slightly different from what actually gets used
    # # while test_datasize during training is number of frames to denoise during optional visualization/saving after each epoch (so I assign it variable name num_frames_of_each_tif_to_denoise_for_visualization_during_training)
    # # while test_datasize during testing is number of frames to denoise using the model (so I assign it variable name num_frames_of_each_tif_to_denoise)
    # # and select_img_num is number of frames in each tif file to include in training, counted from the beginning of each stack (tif) (this param is not used in testing, but test_datasize is analogous)

    # # overlap factor applies to patch x and y, but not patch t
    # # patch t spacing is based on how many xy patches there are in each frame, train_datasets_size, and patch_t, it is automatically calculated to evenly distribute the patches in time

    # # deepcad's demo "best model" for int16 stack shape (6955,492,492) is train_datasets_size = 6000, n_epochs = 20, patch_x,y,t = 150, overlap_factor = 0.4
    # # here is their demo data:
    # # fn_demo ='fish_localbrain' # select the demo file you want to train (e.g. 'ATP_3D', 'fish_localbrain', 'NP_3D', ...)
    # # pth_demo, _ = download_demo(download_filename=fn_demo)
    # # demodata = imread(pth_demo + '/fish_localbrain.tif')

    # # the model should not converge (loss should not decrease across epochs) since the source and target are both original noisy images
    # # so do not use the loss to guide you in tuning parameters. just inspect the results.
    # # in general the last epoch is the one to use,
    # # but if it looks just smoothed, it's underfit (so try more epochs or larger train_dataset_size), and if it looks too sharp/punctate, it's overfit (so try fewer epochs or smaller train_dataset_size)
    # # epochs are continuous (not independent), so if training is interrupted, reload the last completed epoch on the .pth file and resume training (code is not yet written to do this, see above)

    for pth_trainset, pth_testset in zip(pth_trainset_all, pth_testset_all):

          print(pth_trainset)

          n_epochs = 5  # number of training epochs (loss is continuous across patches and epochs - epochs and patches are not independent)
          epochs_choose = [3, 5] #[int(n_epochs/2), n_epochs] #list, which training epochs (which states of the model) to use for testing (denoising), for now choosing last and middle, and inspecting for overfit or underfit
          train_datasets_size = 6000 #how many 3d xyt patches to train on, which is slightly different from what actually gets used
          select_img_num = 1e10 # number of images to take from the beginning of each stack (make larger than Lt use the full stack)
          patch_x = 110 #int(np.ceil(Lx/4)) #extent of patch in x
          patch_y = 110 #int(np.ceil(Ly/4)) #extent of patch in y
          overlap_factor = 0.9        # the overlap factor between two adjacent patches
          patch_t = 300 #extent of patch in t
          intensity_scale_factor = 1 # the factor for image intensity scaling
          num_frames_of_each_tif_to_denoise_for_visualization_during_training = 400 #for the optional inference visualization if save_test_images_per_epoch or visualize_images_per_epoch is True, and the code defaults to taking this number after the first 50 frames for display/save
          num_frames_of_each_tif_to_denoise = Lt #this is number of frames of each tif to be tested (denoised); just make this the length of the stack (or greater) to get the whole stack denoised
          GPU = '0'                   # the index of GPU you will use (e.g. '0', '0,1', '0,1,2')
          num_workers = 0             # if you use Windows system, set this to 0.
          save_test_images_per_epoch = True  # whether to save result images after each epoch

          train_dict = {
              # dataset dependent parameters
              'patch_x': patch_x,                          # the width of 3D patches
              'patch_y': patch_y,                          # the height of 3D patches
              'patch_t': patch_t,                          # the time dimension (frames) of 3D patches
              'overlap_factor': overlap_factor,             # the factor for image intensity scaling
              'scale_factor': intensity_scale_factor,      # the factor for image intensity scaling
              'select_img_num': select_img_num, # number of images to take from the beginning of each stack (make larger than Lt use the full stack)
              'train_datasets_size': train_datasets_size,  # datasets size for training (how many 3D patches)
              'test_datasize': num_frames_of_each_tif_to_denoise_for_visualization_during_training,
              'datasets_path': pth_trainset,             # folder containing files for training
              'pth_dir': pth_trainset,                   # the path for pth file (saved models) and optional test images saved after each epoch if save_test_images_per_epoch=True

              # network related parameters
              'n_epochs': n_epochs,                          # the number of training epochs
              'lr': 0.00005,                                 # learning rate
              'b1': 0.5,                                     # Adam: beta1
              'b2': 0.9,                                   # Adam: beta2
              'fmap': 16,    # model complexity, 16 by default, deepcad author says it should not require adjustment
              'GPU': GPU,                                    # GPU index
              'num_workers': num_workers,                    # if you use Windows system, set this to 0.
              'visualize_images_per_epoch': False,                       # whether to show result images after each epoch
              'save_test_images_per_epoch': save_test_images_per_epoch,  # whether to save result images after each epoch
              'colab_display': True #if colab_display is true and save_test_images_per_epoch is false, it will error between training and testing
          }

          tc = training_class(train_dict)
          tc.run()


    ############################################# TEST (DENOISE) RECORDINGS WITH CHOSEN MODEL ##########################

          # deepcad defaults to denoising all tifs in datasets_path with all models (.pth files) in folder denoise_model
          # and will output denoised versions of those tifs and save in output_dir
          # but, to override default behavior, here i move all pth files except those listed in epochs_choose (which can still be all of them)

          pth_para = natsorted(glob.glob(pth_testset + '/' + '*.yaml'))[-1] #yaml file contains parameters used for training, to be loaded and reused for testing, since there is one yaml per folder, taking the last glob output takes the yaml in the most recent folder (the current run because of datetime in name)
          pth_pth_all = natsorted(glob.glob(pth_testset + '/' + '*.pth')) #paths to pth files (trained models, one for each epoch )

          pth_pth_keep_pattern = []
          for eci, epoch_choose in enumerate(epochs_choose):
            pth_pth_keep_pattern.append('E_' + "{:02d}".format(epoch_choose) + '_*.pth')
          fldr_unused_pth = pth_fldr_pth + '/' + 'unused_pth_files/' #folder for the pth files you don't want to use for testing

          for ppi,pth_pth in enumerate(pth_pth_all): #make sure there aren't multiple train folders before you move pth files below
            pth_fldr_pth = '/'.join(pth_pth.split('/')[:-1])
            if ppi>0 and pth_fldr_pth != pthcheck_prev:
              print(pthcheck_prev)
              print(pth_fldr_pth)
              raise Exception("multiple training folders, not allowed until resume training functionality exists")
            pthcheck_prev = pth_fldr_pth

          for pth_pth in pth_pth_all: #move all pth files besides the ones you want to test with
            fn_pth = pth_pth.split('/')[-1]
            if not any(fnmatch.fnmatch(fn_pth, pat+'*') for pat in pth_pth_keep_pattern):
              if not os.path.exists(fldr_unused_pth):
                os.mkdir(fldr_unused_pth)
              shutil.move(pth_pth, fldr_unused_pth) #move all pth files besides the ones you want to test with

          fldr_pth = pth_fldr_pth.split('/')[-1] #folder with all the pth files


          with open(pth_para, "r") as stream: #read the params from training to apply to testing
              patch_t = train_dict['patch_t']
              patch_x = train_dict['patch_x']
              patch_y = train_dict['patch_y']
              fmap = train_dict['fmap']
              overlap_factor = train_dict['overlap_factor']

          test_dict = {
              # dataset dependent parameters
              'patch_x': patch_x,               # the width of 3D patches
              'patch_y': patch_y,               # the height of 3D patches
              'patch_t': patch_t,               # the time dimension (frames) of 3D patches
              'overlap_factor':overlap_factor,    # overlap factor
              'scale_factor': intensity_scale_factor, # the factor for image intensity scaling
              'test_datasize': num_frames_of_each_tif_to_denoise,     # the number of frames in each tif to be denoised/tested, measured from start
              'datasets_path': pth_trainset,     # folder containing all files to be tested
              'pth_dir': pth_trainset,                 # pth file root path
              'denoise_model' : fldr_pth,    # A folder containing all models (pth files) to be tested
              'output_dir' : pth_trainset,         # result file root path
              # network related parameters
              'fmap': fmap,                         # number of feature maps
              'GPU': GPU,                         # GPU index
              'num_workers': num_workers,         # if you use Windows system, set this to 0.
              'visualize_images_per_epoch': False, # whether to display inference performance after each epoch
              'save_test_images_per_epoch': save_test_images_per_epoch, # whether to save inference image after each epoch in pth path
              'colab_display': True #if colab_display is true and save_test_images_per_epoch is false, it will error between training and testing
          }

          tc = testing_class(test_dict)
          tc.run()





In [14]:
#@title plot denoised movie

if do_denoise:
    
  %matplotlib inline

  filename_add = 'f16'

  for pth_trainset, pth_testset in zip(pth_trainset_all, pth_testset_all):

    outtiff_path1 = natsorted(glob.glob(os.path.join(pth_trainset, '2023*', '*.tif')))

    for fnind, fn_display in enumerate(outtiff_path1):
      if fnind==3:
        print(fn_display)
        fntmp = fn_display.split('/')[-1].replace('.tif', '_output.tif')
        print(fntmp)
        outtiff_path2 = natsorted(glob.glob(os.path.join(pth_trainset, 'DataFolderIs_*', 'E_*', fntmp)))
        for fn_display2 in outtiff_path2:
          print(fn_display2)
          # display_length = 300
          # norm_min_percent = 0
          # norm_max_percent = 100
          # cropflag = 1
          # plot_denoised_movie(filename = fn_display2, filename_add = filename_add, cropflag=cropflag, display_length=display_length, norm_min_percent = norm_min_percent, norm_max_percent = norm_max_percent)

          # cropflag = 0
          # plot_denoised_movie(filename = fn_display, filename_add = filename_add, cropflag=cropflag, display_length=display_length, norm_min_percent = norm_min_percent, norm_max_percent = norm_max_percent)



In [15]:
#@title load registered movie (if necessary) and create crop box for extraction

region_extraction = 'pb' #string not list of string (ie just do one region in ipynb bc loop is not really convenient in this notebook)
use_denoised = 1

if do_extraction:

    if use_denoised:

        fn_dn_add = ''
        pth_tif_dn = pth_tif_dn + fn_dn_add
        do_volume = 1 #whether to use data from denoising that operated on entire volume, or z stack subset (must exist, ie must match a do_volume value that was previously used/saved at some point in denoise.py)
        epoch_choose = 5 #which epoch to stitch (must exist, ie must be one of epochs_choose in denoise.py)
        force_stitch = 0 #stitch regardless of whether the file already exists (e.g. to use a different run)
        if not os.path.isfile(pth_tif_dn) or force_stitch:
            stitch_denoised_slices(pth_denoising, fn_prefix, pth_tif_dn, md['dims'], do_volume, epoch_choose) #stitch together denoised slices (tyx) into original size (tzyx)
        pth_exin = pth_tif_dn

    else:
        pth_exin = pth_tif_reg

    Y = imread(pth_exin).astype('float32')
    Y = Y.reshape(md['dims'])
    Y = np.transpose(Y, (0, 3, 2, 1)) #put in order t x y z
    print(Y.shape)

    create_croplim = 0

    try:

        fn_croplim_pattern = pth_prefix + '_' + region_extraction + '*_croplim_.npy' #find file matching fov subregion with some crop lim
        fn_croplim = glob.glob(fn_croplim_pattern)
        if len(fn_croplim) > 1:
            raise Exception("too many crop files")
        with open(fn_croplim[0], 'rb') as fnc:
            croplim = np.load(fnc)
        limits_str = str(croplim[0]) + '_' + str(croplim[1]) + '_' + str(croplim[2]) + '_' + str(croplim[3]) + '_' + str(croplim[4]) + '_' + str(croplim[5]) + '_' + str(croplim[6]) + '_' + str(croplim[7])

    except:

        if region_extraction == 'fullfov':

            croplim = np.asarray((1, dims[0], 1, dims[3], 1, dims[2], 1, dims[1])).astype(int)
            limits_str = '1_' + str(dims[0]) + '_1_' + str(dims[3]) + '_1_' + str(dims[2]) + '_1_' + str(dims[1])
            fn_crop_lim = pth_prefix + '_' + region_extraction + '_' + limits_str + '_croplim_.npy'
            with open(fn_crop_lim, 'wb') as fncrop:
                np.save(fncrop, croplim)

        else:

            print("crop fov for " + region_extraction)
            create_croplim = 1
            Ymt = np.mean(Y, axis = 0)
            if Ymt.shape[-1]==1: #only do z slice selection if the movie is volumetric 4d
              zlimits = (1,1)
              Ymtz = np.mean(Ymt, axis = 2)
            else:
              zlimits = select_fov_z(Ymt)
              Ymtz = np.mean(Ymt[:,:,zlimits[0]-1:zlimits[1]-1], axis = 2)
            select_fov_xy(Ymtz)



(3047, 256, 140, 15)


In [16]:
#@title crop fov for extraction

if do_extraction:  #had to put this here (in new cell) to get interactive plots to work in above cell

  if create_croplim:

    fn_crop_lim_xy = pth_tif_reg[0][:-4] + 'croplimxytmp_.npy'
    with open(fn_crop_lim_xy, 'rb') as fnc:
      croplimxy = np.load(fnc)
    ylimits = (croplimxy[0], croplimxy[1])
    xlimits = (croplimxy[2], croplimxy[3])
    tlimits = (1, md['dims'][0])
    croplim = np.asarray((tlimits + xlimits + ylimits + zlimits)).astype(int) #make them 1-indexed for matlab later
    limits_str = str(croplim[0]) + '_' + str(croplim[1]) + '_' + str(croplim[2]) + '_' + str(croplim[3]) + '_' + str(croplim[4]) + '_' + str(croplim[5]) + '_' + str(croplim[6]) + '_' + str(croplim[7])
    fn_crop_lim = pth_prefix + '_' + region_extraction + '_' + limits_str + '_croplim_.npy'
    with open(fn_crop_lim, 'wb') as fncrop:
        np.save(fncrop, croplim)

  slt = slice(croplim[0]-1, croplim[1], 1) #croplim are 1-indexed, and the second/upper is not included
  slx = slice(croplim[2]-1, croplim[3], 1)
  sly = slice(croplim[4]-1, croplim[5], 1)
  slz = slice(croplim[6]-1, croplim[7], 1)
  indices_crop = [slt, slx, sly, slz]
  Ycrop = Y[tuple(indices_crop)]
  Y = None




In [17]:
#@title start cluster

if do_cluster:

    if 'dview' in locals(): cm.stop_server(dview=dview)
    cc, dview, n_processes = cm.cluster.setup_cluster(backend='ipyparallel', n_processes=None, single_thread=False)

In [18]:
#@title memmap and load movie for extraction

if do_extraction:

  pth_tif_ex = pth_exin[:-4] + region_extraction + '_' + limits_str + '_cmex_tmp_.tif'
  imwrite(pth_tif_ex, Ycrop.squeeze()) #squeeze in case 3d . . . also must imwrite it to memmap it, and must memmap it to use patches in extraction
  basename_memap = pth_tif_ex.split('/')[-1][:-4]
  border_to_0 = 0 #if mc.border_nan == 'copy' else mc.border_to_0
  fn_mmap_ex = cm.save_memmap([pth_tif_ex], base_name=basename_memap, order='C', border_to_0=border_to_0, dview=dview) # exclude borders
  os.remove(pth_tif_ex)
  Ycrop, dims_spatial_ex, dim_time_ex = cm.load_memmap(fn_mmap_ex) #if 3d mmap should be 3d, but Ycrop gets singleton 4th dim (z) added below so the code is more readable
  Ycrop = np.reshape(Ycrop.T, [dim_time_ex] + list(dims_spatial_ex), order='F')

  if len(Ycrop.shape)==3: #if it's not volumetric
      Ycrop = Ycrop[...,np.newaxis] #add singleton 4th dim (z) so the code is more readable

  print(Ycrop.shape)


(3047, 127, 36, 8)


In [19]:
#@title extract

index_extraction_param_set = 0 #redefined here in notebook (originally defined in "choose files" cell, but that's far away and we're not looping in notebook)
#index_extraction_param_set = 'default'

if do_extraction:

    if index_extraction_param_set == 'default':
        index_extraction_param_set_new = ['default'] #make it iterable with brackets
    elif index_extraction_param_set<0: #if negative, initiate loop over extraction params here, range [0 - index_extraction_param_set]
        manual_start_ind = 0
        index_extraction_param_set_new = np.arange(manual_start_ind, -index_extraction_param_set)
    else: #if positive, just the one extraction param whose index matches index_extraction_param_set
        index_extraction_param_set_new = [index_extraction_param_set]

    print("index_extraction_param_set NEW " + str(index_extraction_param_set_new))

    for ii in index_extraction_param_set_new:

        try: #try, since some param sets will error

          opts_dict, indices_ex, fnadd = configs(index_extraction_param_set = ii, fnames = fn_mmap_ex, md = md, do_planar_extraction = do_planar_extraction, dims_spatial_ex = dims_spatial_ex) #param set for extraction
          opts = cnmf.params.CNMFParams(params_dict=opts_dict)

          if do_planar_extraction: #adjust images and some params for planar
              sliceindz = np.arange(Ycrop.shape[3])
              dims_roimask_spatial = (dims_spatial_ex[0], dims_spatial_ex[1])
          else:
              sliceindz = [np.arange(Ycrop.shape[3])] #all slices in one list (not planar)
              dims_roimask_spatial = (dims_spatial_ex[0], dims_spatial_ex[1], dims_spatial_ex[2])

          countz = 0
          for si in sliceindz: #for each slice (or all slices if do_planar_extraction = false)

              cnm = None
              cnm2 = None

              if do_planar_extraction:
                  print("PLANAR EXTRACTION FOR SLICE " + str(si))
                  images_sliced = Ycrop[:,:,:,si]
              else:
                  print("3D EXTRACTION FOR ALL SLICES")
                  images_sliced = Ycrop #can't .copy() for some reason, for 3d extraction keep all images for loop over ii

              cnm = cnmf.CNMF(n_processes, params=opts, dview=dview)
              cnm = cnm.fit(images_sliced, indices = indices_ex)

              cnm.estimates.evaluate_components(images_sliced, cnm.params, dview=dview)
              print(('NUMGOOD ' + str(len(cnm.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm.estimates.idx_components_bad))))

              cnm.estimates.select_components(use_object=True, save_discarded_components=False)

              ######################  REFIT  ############################
              cnm2 = cnm.refit(images_sliced)

              cnm2.estimates.evaluate_components(images_sliced, cnm2.params, dview=dview)
              print(('REFIT: NUMGOOD ' + str(len(cnm2.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm2.estimates.idx_components_bad))))

              cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=False) #use_residuals=False to not include residuals in traces for dff computation (default)
              dff_residfalse = cnm2.estimates.F_dff
              cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=True) #use_residuals=True to include residuals in traces for dff computation
              dff_residtrue = cnm2.estimates.F_dff
              cnm2.estimates.select_components(use_object=True, save_discarded_components=False)


              if countz==0:

                  numroi_stack_pad = cnm2.estimates.A.shape[-1]
                  dims_roimask_stack = ( dims_roimask_spatial + (numroi_stack_pad, ) )
                  dims_roimask_b_stack = ( dims_roimask_spatial + (cnm2.estimates.b.shape[-1], ) )
                  dims_timeseries_stack = ( numroi_stack_pad, cnm2.estimates.C.shape[1] )

                  stack_masks = np.zeros(dims_roimask_stack + (len(sliceindz), ) )
                  stack_masks_b = np.zeros(dims_roimask_b_stack + (len(sliceindz), ) )
                  stack_c = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
                  # stack_yra = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
                  stack_s = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
                  stack_df = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
                  # stack_dfr = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
                  stack_snr = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
                  stack_rval = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
                  # stack_idx = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
                  # stack_idx_bad = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )

              numroi_slice = cnm2.estimates.A.shape[-1]
              numroi_b_slice = cnm2.estimates.b.shape[-1]
              dims_mask_slice = (dims_roimask_spatial + (numroi_slice, ) )
              dims_mask_b_slice = (dims_roimask_spatial + (numroi_b_slice, ) )

              stack_masks[..., :numroi_slice, countz] = np.reshape(cnm2.estimates.A.toarray(), dims_mask_slice, order='F')
              stack_masks_b[..., :numroi_b_slice, countz] = np.reshape(cnm2.estimates.b, dims_mask_b_slice, order='F')
              stack_c[:numroi_slice,:,countz] = cnm2.estimates.C
              # stack_yra[:numroi_slice,:,countz] = cnm2.estimates.YrA
              stack_s[:numroi_slice,:,countz] = cnm2.estimates.S
              stack_df[:numroi_slice,:,countz] = dff_residfalse
              # stack_dfr[:numroi_slice,:,countz] = dff_residtrue
              stack_snr[:numroi_slice,countz] = cnm2.estimates.SNR_comp
              stack_rval[:numroi_slice,countz] = cnm2.estimates.r_values
              #stack_idx[:cnm2.estimates.idx_components.shape[0],countz] = cnm2.estimates.idx_components
              #if cnm2.estimates.idx_components_bad.shape==(1,):
              #    stack_idx_bad[:cnm2.estimates.idx_components_bad.shape[0],countz] = cnm2.estimates.idx_components_bad

              countz = countz + 1

          log_files = glob.glob('*_LOG_*')
          for log_file in log_files:
              os.remove(log_file)

          mdict = {}
          mdict['roimasks'] = stack_masks.astype('float32')
          mdict['roimasks_b'] = stack_masks_b.astype('float32')
          mdict['C'] = stack_c.astype('float32')
          # mdict['YrA'] = stack_yra.astype('float32')
          mdict['S'] = stack_s.astype('float32')
          mdict['dff'] = stack_df.astype('float32')
          # mdict['dffr'] = stack_dfr.astype('float32')
          mdict['snr'] = stack_snr.astype('float32')
          mdict['rval'] = stack_rval.astype('float32')
          #mdict['idx'] = stack_idx
          #mdict['idxbad'] = stack_idx_bad

          if np.any(stack_masks):
            pth_mat_ex = pth_tif_ex[:-8] + fnadd + '_rois_.mat'
          else:
            mdict = {}
            print("norois")
            pth_mat_ex = pth_tif_ex[:-8] + fnadd + '_rois_NOROIS_.mat'

        except Exception as error:

            mdict = {}
            pth_mat_ex = pth_tif_ex[:-8] + fnadd + '_rois_FAILURE_.mat'
            print("An exception occurred:", type(error).__name__, "-", error)


        sio.savemat(pth_mat_ex, mdict)

        if dview is not None: cm.stop_server(dview=dview)




index_extraction_param_set NEW [0]
indexing into param set
index_extraction_param_set is 0 with filename string 2_1_0.9_9_1_1_50_1000_3_graph_3dex
3D EXTRACTION FOR ALL SLICES
entering fit
 , gsiz is . . . 
(5, 5, 3)
skip refinement is . . . 
False
max_merge_area is . . . 
None
run patches cnmf
 , gsiz is . . . 
(5, 5, 3)
 , rfs is . . . 
[9, 9, 9]
 , stride is . . . 
[7, 7, 7]
in extract patch coords
numpatches is . . . 
33
loading memmap filename
/Users/wienecke/Documents/ambrose/stacks/20230627-2_D05_syt7f_018_syt7f/20230627_2_2_cmrg_dcdn_pb_1_3047_48_174_4_39_2_9_cmex_tmp_d1_127_d2_36_d3_8_order_C_frames_3047.mmap
(3047, 127, 36, 8)
entering fit
 , gsiz is . . . 
(5, 5, 3)
USING INDICES TO SLICE FOV
[slice(None, None, None), slice(None, None, None), slice(None, None, None), slice(None, None, None)]
(3047, 19, 19, 8)
(3047, 19, 19, 8)
skip refinement is . . . 
False
max_merge_area is . . . 
None
initializing, gsiz is . . . 
(5, 5, 3)
entering initialization.py, Y shape is . . . 
(19

      304729 [components_evaluation.py:classify_components_ep():246] [67773] Component 34 is only active jointly with neighboring components. Space correlation calculation might be unreliable.


evaluate_components_placeholder
evaluate components line 352 function
NUMGOOD 74 NUMBAD 0
*** Variable cnn_preds has not the same number of components as A ***
entering fit
 , gsiz is . . . 
(5, 5, 3)
skip refinement is . . . 
False
max_merge_area is . . . 
None
update spatialman first
(36576, 74)
entering update_spatial_components (pursuit) in spatial.py
entering determine search location 
entering regression_ipyparallel (pursuit) in spatial.py
entering regression_ipyparallel (pursuit) in spatial.py
prethresh there are 0 empty spatial component(s)
removing 1237 pixels with least energy, of 36576 total pixels
removing 895 pixels with least energy, of 36576 total pixels
removing 978 pixels with least energy, of 36576 total pixels
removing 544 pixels with least energy, of 36576 total pixels
removing 1595 pixels with least energy, of 36576 total pixels
removing 414 pixels with least energy, of 36576 total pixels
removing 1099 pixels with least energy, of 36576 total pixels
removing 1421 p

In [20]:
#@title plot rois 

#cnm.estimates.nb_view_components_3d(image_type='corr', dims=dims_spatial_ex, Yr=images_sliced, denoised_color='red', max_projection=True);
#cnm.estimates.nb_view_components_3d(image_type='mean', axis = 2, dims=dims_spatial_ex, Yr=None, denoised_color='red', max_projection=False);
#cnm2.estimates.nb_view_components_3d(image_type='corr', dims=dims_spatial_ex, Yr=images_sliced, denoised_color='red', max_projection=True);
cnm2.estimates.nb_view_components_3d(image_type='mean', axis = 2, dims=dims_spatial_ex, Yr=None, denoised_color='red', max_projection=False);

# cnm2.estimates.SNR_comp
# cnm2.estimates.r_values

In [21]:
#@title plot responses and residuals

# indies = np.arange(1,600)
# # plt.plot(cnm2.estimates.C[1,indies])

# # plt.plot(cnm2.estimates.R[1,indies])
# # plt.plot(dff_residfalse[1,indies])
# # plt.plot(dff_residtrue[1,indies])

# data1 = cnm2.estimates.C[0,indies]
# data2 = cnm2.estimates.R[0,indies]
# data3 = data1+data2
# data4 = dff_residfalse[0,indies]
# data5 = dff_residtrue[0,indies]
# t = indies

# fig, ax1 = plt.subplots()

# color1 = 'tab:blue'
# color2 = 'tab:red'
# color3 = 'tab:pink'
# color4 = 'tab:green'
# color5 = 'tab:purple'
# ax1.set_xlabel('time (s)')
# ax1.set_ylabel('exp', color='black')
# #ax1.plot(t, data1, color=color1)
# #ax1.plot(t, data2, color=color2)
# ax1.plot(t, data3, color=color3)
# ax1.tick_params(axis='y', labelcolor='black')
# ylim1 = ax1.get_ylim()

# ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

# ax2.set_ylabel('sin', color='black')  # we already handled the x-label with ax1
# #ax2.plot(t, data4, color=color4)
# #ax2.plot(t, data5, color=color5)
# ax2.tick_params(axis='y', labelcolor='black')
# #ax2.set_ylim([ylim1[0], ylim1[1]])

# fig.tight_layout()  # otherwise the right y-label is slightly clipped
# plt.show()






In [22]:
#@title extract again (now integrated above)

# if do_extraction:

#   try:

#     cnm2 = cnm.refit(images_sliced)

#     cnm2.estimates.evaluate_components(images_sliced, cnm2.params, dview=dview)
#     print(('REFIT: NUMGOOD ' + str(len(cnm2.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm2.estimates.idx_components_bad))))

#     cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=False) #use_residuals=False to not include residuals in traces for dff computation (default)
#     dff_residfalse = cnm2.estimates.F_dff
#     cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=True) #use_residuals=True to include residuals in traces for dff computation
#     dff_residtrue = cnm2.estimates.F_dff
#     cnm2.estimates.select_components(use_object=True, save_discarded_components=False)

#   except:

#     mdict = {}
#     pth_mat_ex = [pth_tif_ex[:-8] + fnadd + '_rois_FAILURE_.mat']


In [23]:
#@title plots (now integrated above)

# if do_extraction:

#   if do_planar_extraction:

#       Cn = cm.local_correlations(images_sliced.transpose(1,2,0))
#       Cn[np.isnan(Cn)] = 0
#       print('you may need to change the data rate to generate nb_view_components: use jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10 before opening jupyter notebook')

#       cnm.estimates.plot_contours_nb(img=Cn) #img=None for mean projection

#       cnm2.estimates.plot_contours_nb(img=Cn, idx=cnm2.estimates.idx_components)

#       cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components)

#       if len(cnm2.estimates.idx_components_bad) > 0:
#         cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components_bad)
#       else:
#         print("No components were rejected.")

#       cnm.estimates.play_movie(images_sliced, q_max=99.9, gain_res=2, magnification=2, bpx=True, include_bck=False, save_movie=True)

#       A2 = cnm.estimates.A.toarray().reshape(opts.data['dims'] + (-1,), order='F').transpose([2, 0, 1])
#       Nc = A2.shape[0]
#       grid_shape = (np.ceil(np.sqrt(Nc/2)).astype(int), np.ceil(np.sqrt(Nc*2)).astype(int))
#       plt.figure(figsize=np.array(grid_shape[::-1])*1.5)
#       plt.imshow(montage(A2, rescale_intensity=True, grid_shape=grid_shape))
#       plt.axis('off')

#   else:

#       cnm.estimates.nb_view_components_3d(image_type='mean', dims_spatial=dims_spatial_ex, axis=2)



In [24]:
#@title save (now integrated above)

# if do_extraction:

#   if countz==0: #THIS IS POINTLESS WITHOUT THE LOOP, COUNTZ IS ALWAYS 0 (NO LOOP IN IPYNB CURRENTLY)

#           padnum = 0
#           padnum_b = 0
#           numroi_stack_pad = cnm2.estimates.A.shape[-1] + padnum #extra since it can vary a little across fits (even above input k)
#           dims_roimask_stack = ( dims_roimask_spatial + (numroi_stack_pad, ) )
#           dims_roimask_b_stack = ( dims_roimask_spatial + (cnm2.estimates.b.shape[-1] + padnum_b, ) )
#           dims_timeseries_stack = ( numroi_stack_pad, cnm2.estimates.C.shape[1] )

#           stack_masks = np.zeros(dims_roimask_stack + (len(sliceindz), ) )
#           stack_masks_b = np.zeros(dims_roimask_b_stack + (len(sliceindz), ) )
#           stack_c = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
#           # stack_yra = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
#           stack_s = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
#           stack_df = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
#           # stack_dfr = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
#           stack_snr = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
#           stack_rval = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
#           # stack_idx = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
#           # stack_idx_bad = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )

#       numroi_slice = cnm2.estimates.A.shape[-1]
#       numroi_b_slice = cnm2.estimates.b.shape[-1]
#       dims_mask_slice = (dims_roimask_spatial + (numroi_slice, ) )
#       dims_mask_b_slice = (dims_roimask_spatial + (numroi_b_slice, ) )

#       stack_masks[..., :numroi_slice, countz] = np.reshape(cnm2.estimates.A.toarray(), dims_mask_slice, order='F')
#       stack_masks_b[..., :numroi_b_slice, countz] = np.reshape(cnm2.estimates.b, dims_mask_b_slice, order='F')
#       stack_c[:numroi_slice,:,countz] = cnm2.estimates.C
#       # stack_yra[:numroi_slice,:,countz] = cnm2.estimates.YrA
#       stack_s[:numroi_slice,:,countz] = cnm2.estimates.S
#       stack_df[:numroi_slice,:,countz] = dff_residfalse
#       # stack_dfr[:numroi_slice,:,countz] = dff_residtrue
#       stack_snr[:numroi_slice,countz] = cnm2.estimates.SNR_comp
#       stack_rval[:numroi_slice,countz] = cnm2.estimates.r_values
#       #stack_idx[:cnm2.estimates.idx_components.shape[0],countz] = cnm2.estimates.idx_components
#       #if cnm2.estimates.idx_components_bad.shape==(1,):
#       #    stack_idx_bad[:cnm2.estimates.idx_components_bad.shape[0],countz] = cnm2.estimates.idx_components_bad

#       countz = countz + 1

#   log_files = glob.glob('*_LOG_*')
#   for log_file in log_files:
#       os.remove(log_file)

#   mdict = {}
#   mdict['roimasks'] = stack_masks.astype('float32')
#   mdict['roimasks_b'] = stack_masks_b.astype('float32')
#   mdict['C'] = stack_c.astype('float32')
#   # mdict['YrA'] = stack_yra.astype('float32')
#   mdict['S'] = stack_s.astype('float32')
#   mdict['dff'] = stack_df.astype('float32')
#   # mdict['dffr'] = stack_dfr.astype('float32')
#   mdict['snr'] = stack_snr.astype('float32')
#   mdict['rval'] = stack_rval.astype('float32')
#   #mdict['idx'] = stack_idx
#   #mdict['idxbad'] = stack_idx_bad

#   if np.any(stack_masks):
#       pth_mat_ex = pth_tif_ex[:-8] + fnadd + '_rois_.mat'
#   else:
#       mdict = {}
#       print("norois")
#       pth_mat_ex = pth_tif_ex[:-8] + fnadd + '_rois_NOROIS_.mat'

#   sio.savemat(pth_mat_ex, mdict)

#   if dview is not None: cm.stop_server(dview=dview)